# PubMedQA Semi-Supervised Self-Training Pipeline

**Model**: `microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext`  
**Task**: 3-class classification — `yes / no / maybe`  
**Strategy**: Iterative pseudo-label self-training on PQA-U (61k unlabeled)

### Pipeline stages
1. Install & imports
2. Configuration
3. Data loading (PQA-L, PQA-U, PQA-A)
4. Preprocessing & tokenization
5. Dataset classes & DataLoaders
6. Model (PubMedBERT classifier)
7. Supervised training on PQA-L
8. Self-training loop (pseudo-labeling)
9. Final evaluation & report

---
## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets scikit-learn torch

---
## 2. Imports

In [ ]:
import random
import logging
import warnings
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset, Subset
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

PyTorch  : 2.10.0+cu128
Device   : cuda
GPU      : Tesla T4


---
## 3. Configuration

Edit this cell to change any hyperparameter.

In [ ]:
class Config:

    MODEL_NAME  = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
    NUM_LABELS  = 3
    LABEL_MAP   = {"yes": 0, "no": 1, "maybe": 2}
    ID2LABEL    = {0: "yes", 1: "no", 2: "maybe"}


    MAX_LENGTH  = 512
    VAL_SPLIT   = 0.2
    BATCH_SIZE  = 16


    LR           = 2e-5
    EPOCHS       = 5
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01
    GRAD_CLIP    = 1.0


    ST_ROUNDS          = 3
    CONFIDENCE_THRESH  = 0.90
    ST_EPOCHS          = 3
    MAX_PSEUDO_RATIO   = 5.0
    ANNEAL_STEP        = 0.05
    ANNEAL_MIN         = 0.70
    MIN_DELTA          = 0.002


    USE_PQA_A       = True
    PQA_A_EPOCHS    = 2
    PQA_A_MAX_SAMP  = 50_000


    SEED       = 42
    OUTPUT_DIR = Path("outputs")
    DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    FP16       = torch.cuda.is_available()


cfg = Config()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.SEED)
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Config ready | device={cfg.DEVICE} | fp16={cfg.FP16}")
print(f"Model: {cfg.MODEL_NAME}")

Config ready | device=cuda | fp16=True
Model: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext


---
## 4. Load PubMedQA splits

| Split | Key | Size | Labels |
|---|---|---|---|
| PQA-L | `pqa_labeled` | 1,000 | Expert yes/no/maybe |
| PQA-U | `pqa_unlabeled` | 61,333 | None |
| PQA-A | `pqa_artificial` | 211,269 | Auto-generated (weak) |

In [ ]:
pqa_l = load_dataset("pubmed_qa", "pqa_labeled", split="train")
print(f"  PQA-L : {len(pqa_l)} samples")
print(f"  Fields: {list(pqa_l[0].keys())}")


from collections import Counter
label_dist = Counter(s["final_decision"] for s in pqa_l)
print(f"  Labels: {dict(label_dist)}")

pqa_u = load_dataset("pubmed_qa", "pqa_unlabeled", split="train")
print(f"  PQA-U : {len(pqa_u)} samples")

if cfg.USE_PQA_A:
    pqa_a = load_dataset("pubmed_qa", "pqa_artificial", split="train")
    print(f"  PQA-A : {len(pqa_a)} samples")
    label_dist_a = Counter(s["final_decision"] for s in pqa_a)
    print(f"  Labels: {dict(label_dist_a)}")
else:
    pqa_a = None
    print("\nPQA-A skipped (USE_PQA_A=False). Set cfg.USE_PQA_A=True to enable weak pre-training.")

  PQA-L : 1000 samples
  Fields: ['pubid', 'question', 'context', 'long_answer', 'final_decision']
  Labels: {'yes': 552, 'no': 338, 'maybe': 110}
  PQA-U : 61249 samples
  PQA-A : 211269 samples
  Labels: {'yes': 196144, 'no': 15125}


### Inspect a sample

In [ ]:
sample = pqa_l[0]
print(f"PUBID         : {sample['pubid']}")
print(f"QUESTION      : {sample['question']}")
print(f"FINAL DECISION: {sample['final_decision']}")
print(f"LONG ANSWER   : {sample['long_answer'][:200]}...")
print(f"\nCONTEXT KEYS  : {list(sample['context'].keys())}")
print(f"NUM PASSAGES  : {len(sample['context']['contexts'])}")
print(f"PASSAGE[0]    : {sample['context']['contexts'][0][:200]}...")

PUBID         : 21645374
QUESTION      : Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
FINAL DECISION: yes
LONG ANSWER   : Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of ...

CONTEXT KEYS  : ['contexts', 'labels', 'meshes', 'reasoning_required_pred', 'reasoning_free_pred']
NUM PASSAGES  : 2
PASSAGE[0]    : Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant co...


---
## 5. Preprocessing & Tokenization

We flatten the context passages into a single string and format as:
```
[CLS] question [SEP] passage1 passage2 ... [SEP]
```

In [ ]:
print(f"Loading tokenizer: {cfg.MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(cfg.MODEL_NAME)
print(f"Vocab size: {tokenizer.vocab_size:,}")


def build_input_text(sample: dict) -> str:
    question = sample["question"].strip()
    passages = " ".join(sample["context"]["contexts"])
    return f"{question} [SEP] {passages}"


def get_label(sample: dict) -> int:
    return cfg.LABEL_MAP.get(sample.get("final_decision", ""), -1)


def tokenize(text: str) -> dict:
    return tokenizer(
        text,
        max_length=cfg.MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )


text = build_input_text(pqa_l[0])
enc  = tokenize(text)
print(f"\nSample input text (first 200 chars): {text[:200]}...")
print(f"input_ids shape   : {enc['input_ids'].shape}")
print(f"attention_mask sum: {enc['attention_mask'].sum().item()} non-padding tokens")

Loading tokenizer: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Vocab size: 30,522

Sample input text (first 200 chars): Do mitochondria play a role in remodelling lace plant leaves during programmed cell death? [SEP] Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponoge...
input_ids shape   : torch.Size([1, 512])
attention_mask sum: 353 non-padding tokens


---
## 6. Dataset classes & DataLoaders

In [ ]:
class LabeledDataset(Dataset):

    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        enc    = tokenize(build_input_text(sample))
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(get_label(sample), dtype=torch.long),
        }


class UnlabeledDataset(Dataset):

    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        enc    = tokenize(build_input_text(sample))
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }


class PseudoLabeledDataset(Dataset):

    def __init__(self, input_ids, attention_masks, labels):
        self.input_ids       = input_ids
        self.attention_masks = attention_masks
        self.labels          = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_masks[idx],
            "labels":         self.labels[idx],
        }


def make_loader(dataset, batch_size, shuffle=False):
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        num_workers=2, pin_memory=torch.cuda.is_available()
    )


def merge_with_pseudo(base_loader, pseudo_ds):
    labeled_ds   = base_loader.dataset
    n_labeled    = len(labeled_ds)
    n_pseudo_max = int(n_labeled * cfg.MAX_PSEUDO_RATIO)

    if len(pseudo_ds) > n_pseudo_max:
        idxs = torch.randperm(len(pseudo_ds))[:n_pseudo_max]
        pseudo_ds = PseudoLabeledDataset(
            [pseudo_ds.input_ids[i]       for i in idxs],
            [pseudo_ds.attention_masks[i] for i in idxs],
            [pseudo_ds.labels[i]          for i in idxs],
        )

    combined = ConcatDataset([labeled_ds, pseudo_ds])
    print(f"  Combined: {n_labeled} labeled + {len(pseudo_ds)} pseudo = {len(combined)} total")
    return make_loader(combined, cfg.BATCH_SIZE, shuffle=True)



full_labeled = LabeledDataset(pqa_l)
n_val        = int(len(full_labeled) * cfg.VAL_SPLIT)
n_train      = len(full_labeled) - n_val
train_ds, val_ds = random_split(
    full_labeled, [n_train, n_val],
    generator=torch.Generator().manual_seed(cfg.SEED)
)

train_loader    = make_loader(train_ds,              cfg.BATCH_SIZE, shuffle=True)
val_loader      = make_loader(val_ds,                cfg.BATCH_SIZE * 2)
unlabeled_loader= make_loader(UnlabeledDataset(pqa_u), cfg.BATCH_SIZE * 4)

print(f"  train    : {n_train} samples ({len(train_loader)} batches)")
print(f"  val      : {n_val} samples ({len(val_loader)} batches)")
print(f"  unlabeled: {len(pqa_u)} samples ({len(unlabeled_loader)} batches)")

# Optional: PQA-A
art_loader = None
if pqa_a is not None:
    art_full   = LabeledDataset(pqa_a)
    cap        = min(cfg.PQA_A_MAX_SAMP, len(art_full))
    art_ds     = Subset(art_full, torch.randperm(len(art_full))[:cap].tolist())
    art_loader = make_loader(art_ds, cfg.BATCH_SIZE, shuffle=True)
    print(f"  artificial: {cap} samples (capped from {len(art_full)})")

  train    : 800 samples (50 batches)
  val      : 200 samples (7 batches)
  unlabeled: 61249 samples (958 batches)
  artificial: 50000 samples (capped from 211269)


---
## 7. Model — PubMedBERT Classifier

In [ ]:
class PubMedBERTClassifier(nn.Module):

    def __init__(self):
        super().__init__()
        self.model = AutoModelForSequenceClassification.from_pretrained(
            cfg.MODEL_NAME,
            num_labels=cfg.NUM_LABELS,
            id2label=cfg.ID2LABEL,
            label2id=cfg.LABEL_MAP,
        )

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(input_ids=input_ids,
                          attention_mask=attention_mask,
                          labels=labels)

    def num_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def save(self, path):
        Path(path).mkdir(parents=True, exist_ok=True)
        self.model.save_pretrained(path)

    @classmethod
    def load(cls, path):
        obj = cls.__new__(cls)
        super(PubMedBERTClassifier, obj).__init__()
        obj.model = AutoModelForSequenceClassification.from_pretrained(str(path))
        return obj


print(f"Loading {cfg.MODEL_NAME} ...")
model = PubMedBERTClassifier().to(cfg.DEVICE)
print(f"Trainable parameters: {model.num_params():,}")

Loading microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- 

Trainable parameters: 109,484,547


---
## 8. Metrics helpers

In [ ]:
def get_accuracy(y_true, y_pred):
    return accuracy_score(y_true, y_pred)

def get_macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro", zero_division=0)


def validate(model, loader):
    model.eval()
    total_loss, all_preds, all_trues = 0.0, [], []
    with torch.no_grad():
        for batch in loader:
            ids   = batch["input_ids"].to(cfg.DEVICE)
            mask  = batch["attention_mask"].to(cfg.DEVICE)
            labs  = batch["labels"].to(cfg.DEVICE)
            with autocast(enabled=cfg.FP16):
                out = model(ids, mask, labels=labs)
            total_loss += out.loss.item()
            all_preds.extend(out.logits.argmax(-1).cpu().tolist())
            all_trues.extend(labs.cpu().tolist())
    return (
        get_accuracy(all_trues, all_preds),
        get_macro_f1(all_trues, all_preds),
        total_loss / len(loader),
    )



---
## 9. Training loop

Reusable function used for supervised training, weak pre-training, and each self-training round.

In [ ]:
def train(
    model,
    train_loader,
    val_loader,
    epochs,
    stage="supervised",
    ckpt_name=None,
    patience=2,
):

    optimizer     = AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    total_steps   = len(train_loader) * epochs
    warmup_steps  = int(total_steps * cfg.WARMUP_RATIO)
    scheduler     = get_linear_schedule_with_warmup(
        optimizer, warmup_steps, total_steps
    )
    scaler        = GradScaler(enabled=cfg.FP16)

    ckpt_path     = cfg.OUTPUT_DIR / f"best_{ckpt_name or stage}.pt"
    best_val_acc  = 0.0
    patience_ctr  = 0

    print(f"\n[{stage.upper()}] epochs={epochs} | "
          f"steps/epoch={len(train_loader)} | warmup={warmup_steps}")

    for epoch in range(1, epochs + 1):
        model.train()
        ep_loss, all_preds, all_trues = 0.0, [], []

        for step, batch in enumerate(train_loader, 1):
            ids  = batch["input_ids"].to(cfg.DEVICE)
            mask = batch["attention_mask"].to(cfg.DEVICE)
            labs = batch["labels"].to(cfg.DEVICE)

            optimizer.zero_grad()
            with autocast(enabled=cfg.FP16):
                out  = model(ids, mask, labels=labs)
                loss = out.loss

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            ep_loss += loss.item()
            all_preds.extend(out.logits.argmax(-1).cpu().tolist())
            all_trues.extend(labs.cpu().tolist())

            if step % max(1, len(train_loader) // 4) == 0:
                print(f"  epoch {epoch} step {step}/{len(train_loader)} "
                      f"loss={loss.item():.4f}")

        t_acc  = get_accuracy(all_trues, all_preds)
        t_f1   = get_macro_f1(all_trues, all_preds)
        v_acc, v_f1, v_loss = validate(model, val_loader)

        print(f"  Epoch {epoch}/{epochs} "
              f"| train_loss={ep_loss/len(train_loader):.4f} "
              f"train_acc={t_acc:.4f} train_f1={t_f1:.4f} "
              f"| val_loss={v_loss:.4f} val_acc={v_acc:.4f} val_f1={v_f1:.4f}")

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            patience_ctr = 0
            torch.save(model.state_dict(), ckpt_path)
            print(f"  ✓ Best val_acc={best_val_acc:.4f} — checkpoint saved")
        else:
            patience_ctr += 1
            if patience_ctr >= patience and stage not in ("pretrain",):
                print(f"  Early stopping (no improvement for {patience} epochs)")
                break

    # Restore best weights
    if ckpt_path.exists():
        model.load_state_dict(torch.load(ckpt_path, map_location=cfg.DEVICE))
        print(f"  Best weights restored (val_acc={best_val_acc:.4f})")
    return best_val_acc

print("Train function ready.")

Train function ready.


---
## 10. (Optional) Weak pre-training on PQA-A

Only runs if `cfg.USE_PQA_A = True`. Pre-trains on 211k auto-labeled samples before fine-tuning on the 1k gold labels.

In [ ]:
if cfg.USE_PQA_A and art_loader is not None:
    print("Pre-training on PQA-A (weak supervision)...")
    train(model, art_loader, val_loader,
          epochs=cfg.PQA_A_EPOCHS, stage="pretrain")
    print("Pre-training complete.")
else:
    print("Skipping PQA-A pre-training (USE_PQA_A=False).")

Pre-training on PQA-A (weak supervision)...

[PRETRAIN] epochs=2 | steps/epoch=3125 | warmup=625
  epoch 1 step 781/3125 loss=0.4552
  epoch 1 step 1562/3125 loss=0.0221
  epoch 1 step 2343/3125 loss=0.3095
  epoch 1 step 3124/3125 loss=0.0307
  Epoch 1/2 | train_loss=0.1806 train_acc=0.9461 train_f1=0.5000 | val_loss=1.3119 val_acc=0.7600 val_f1=0.5203
  ✓ Best val_acc=0.7600 — checkpoint saved
  epoch 2 step 781/3125 loss=0.0029
  epoch 2 step 1562/3125 loss=0.0029
  epoch 2 step 2343/3125 loss=0.0012
  epoch 2 step 3124/3125 loss=0.0175
  Epoch 2/2 | train_loss=0.0851 train_acc=0.9777 train_f1=0.9087 | val_loss=1.4741 val_acc=0.7650 val_f1=0.5249
  ✓ Best val_acc=0.7650 — checkpoint saved
  Best weights restored (val_acc=0.7650)
Pre-training complete.


---
## 11. Supervised training on PQA-L (gold labels)

In [ ]:
print("Supervised training on PQA-L...")
base_val_acc = train(
    model, train_loader, val_loader,
    epochs=cfg.EPOCHS, stage="supervised"
)

base_acc, base_f1, _ = validate(model, val_loader)
print(f"\nBase model → val_acc={base_acc:.4f}  macro_f1={base_f1:.4f}")

Supervised training on PQA-L...

[SUPERVISED] epochs=5 | steps/epoch=50 | warmup=25
  epoch 1 step 12/50 loss=1.7703
  epoch 1 step 24/50 loss=1.9472
  epoch 1 step 36/50 loss=0.9374
  epoch 1 step 48/50 loss=0.5258
  Epoch 1/5 | train_loss=1.2163 train_acc=0.7025 train_f1=0.4897 | val_loss=0.6695 val_acc=0.7350 val_f1=0.5014
  ✓ Best val_acc=0.7350 — checkpoint saved
  epoch 2 step 12/50 loss=0.5659
  epoch 2 step 24/50 loss=0.8058
  epoch 2 step 36/50 loss=0.3245
  epoch 2 step 48/50 loss=0.4756
  Epoch 2/5 | train_loss=0.5945 train_acc=0.7800 train_f1=0.5879 | val_loss=0.6034 val_acc=0.7650 val_f1=0.5815
  ✓ Best val_acc=0.7650 — checkpoint saved
  epoch 3 step 12/50 loss=0.6568
  epoch 3 step 24/50 loss=0.4154
  epoch 3 step 36/50 loss=0.2951
  epoch 3 step 48/50 loss=0.3541
  Epoch 3/5 | train_loss=0.4024 train_acc=0.8612 train_f1=0.7122 | val_loss=0.6572 val_acc=0.7450 val_f1=0.5822
  epoch 4 step 12/50 loss=0.0633
  epoch 4 step 24/50 loss=0.3977
  epoch 4 step 36/50 loss=0.2017

---
## 12. Self-training loop

### Algorithm
```
For round t = 1 ... ST_ROUNDS:
    1. Inference on PQA-U (61k unlabeled)
    2. Accept samples where max(softmax) >= threshold
    3. Merge accepted samples with gold PQA-L labels
    4. Fine-tune model on combined set
    5. If val_acc improves < MIN_DELTA → anneal threshold down by ANNEAL_STEP
```

In [ ]:
def generate_pseudo_labels(model, loader, threshold):
    model.eval()
    ids_buf, mask_buf, label_buf = [], [], []
    label_counts = {0: 0, 1: 0, 2: 0}
    total = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(cfg.DEVICE)
            attn_mask = batch["attention_mask"].to(cfg.DEVICE)

            with autocast(enabled=cfg.FP16):
                out = model(input_ids, attn_mask)

            probs        = torch.softmax(out.logits, dim=-1)
            conf, preds  = probs.max(dim=-1)
            total       += input_ids.size(0)

            for i in conf.ge(threshold).nonzero(as_tuple=True)[0].cpu():
                ids_buf.append(input_ids[i].cpu())
                mask_buf.append(attn_mask[i].cpu())
                label_buf.append(preds[i].cpu().long())
                label_counts[preds[i].item()] += 1

    accepted = len(label_buf)
    dist     = {cfg.ID2LABEL[k]: v for k, v in label_counts.items() if v > 0}
    pct      = 100 * accepted / total if total else 0
    print(f"  Pseudo-labels: {total} candidates → {accepted} accepted "
          f"({pct:.1f}%)  dist={dist}")

    return PseudoLabeledDataset(ids_buf, mask_buf, label_buf), accepted


threshold         = cfg.CONFIDENCE_THRESH
prev_val_acc      = base_acc
current_loader    = train_loader
history           = [{"round": 0, "val_acc": base_acc, "val_f1": base_f1,
                      "threshold": threshold, "pseudo_accepted": 0}]

for round_idx in range(1, cfg.ST_ROUNDS + 1):
    print(f"Self-training round {round_idx}/{cfg.ST_ROUNDS} | threshold={threshold:.2f}")

    pseudo_ds, n_accepted = generate_pseudo_labels(model, unlabeled_loader, threshold)

    if n_accepted == 0:
        new_thresh = max(cfg.ANNEAL_MIN, threshold - cfg.ANNEAL_STEP)
        print(f"  No pseudo labels at threshold={threshold:.2f}. "
              f"Annealing → {new_thresh:.2f}")
        threshold = new_thresh
        continue


    combined_loader = merge_with_pseudo(current_loader, pseudo_ds)


    train(
        model, combined_loader, val_loader,
        epochs=cfg.ST_EPOCHS,
        stage="self_train",
        ckpt_name=f"self_train_r{round_idx}",
    )


    v_acc, v_f1, _ = validate(model, val_loader)
    print(f"  Round {round_idx} → val_acc={v_acc:.4f}  macro_f1={v_f1:.4f}")


    if v_acc - prev_val_acc < cfg.MIN_DELTA and threshold > cfg.ANNEAL_MIN:
        threshold = max(cfg.ANNEAL_MIN, threshold - cfg.ANNEAL_STEP)
        print(f"  Little improvement (Δ={v_acc - prev_val_acc:.4f}). "
              f"Annealing threshold → {threshold:.2f}")

    history.append({
        "round": round_idx, "val_acc": v_acc, "val_f1": v_f1,
        "threshold": threshold, "pseudo_accepted": n_accepted
    })
    prev_val_acc   = v_acc
    current_loader = combined_loader

print(f"\nSelf-training complete. Final threshold={threshold:.2f}")

Self-training round 1/3 | threshold=0.90
  Pseudo-labels: 61249 candidates → 27868 accepted (45.5%)  dist={'yes': 21395, 'no': 6473}
  Combined: 800 labeled + 4000 pseudo = 4800 total

[SELF_TRAIN] epochs=3 | steps/epoch=300 | warmup=90
  epoch 1 step 75/300 loss=0.0258
  epoch 1 step 150/300 loss=0.2377
  epoch 1 step 225/300 loss=0.0008
  epoch 1 step 300/300 loss=0.0035
  Epoch 1/3 | train_loss=0.1384 train_acc=0.9650 train_f1=0.6826 | val_loss=1.1248 val_acc=0.7500 val_f1=0.5697
  ✓ Best val_acc=0.7500 — checkpoint saved
  epoch 2 step 75/300 loss=0.0019
  epoch 2 step 150/300 loss=0.0023
  epoch 2 step 225/300 loss=0.3014
  epoch 2 step 300/300 loss=0.3804
  Epoch 2/3 | train_loss=0.1017 train_acc=0.9754 train_f1=0.7146 | val_loss=1.2873 val_acc=0.7650 val_f1=0.5585
  ✓ Best val_acc=0.7650 — checkpoint saved
  epoch 3 step 75/300 loss=0.0025
  epoch 3 step 150/300 loss=0.0008
  epoch 3 step 225/300 loss=0.1986
  epoch 3 step 300/300 loss=0.2849
  Epoch 3/3 | train_loss=0.0553 trai

---
## 13. Final evaluation & report

In [ ]:
print("Running final evaluation...\n")
model.eval()

all_preds, all_trues, all_probs = [], [], []
with torch.no_grad():
    for batch in val_loader:
        ids  = batch["input_ids"].to(cfg.DEVICE)
        mask = batch["attention_mask"].to(cfg.DEVICE)
        labs = batch["labels"].to(cfg.DEVICE)
        with autocast(enabled=cfg.FP16):
            out = model(ids, mask)
        probs = torch.softmax(out.logits, dim=-1).cpu().numpy()
        all_preds.extend(out.logits.argmax(-1).cpu().tolist())
        all_trues.extend(labs.cpu().tolist())
        all_probs.append(probs)

all_probs = np.concatenate(all_probs, axis=0)

label_names = list(cfg.LABEL_MAP.keys())
label_ids   = list(cfg.LABEL_MAP.values())

print("=" * 60)
print("FINAL EVALUATION REPORT")
print("=" * 60)
print(f"Accuracy  : {get_accuracy(all_trues, all_preds):.4f}")
print(f"Macro-F1  : {get_macro_f1(all_trues, all_preds):.4f}")
print()
print(classification_report(
    all_trues, all_preds,
    target_names=label_names, digits=4
))

print("Confusion Matrix (rows=true, cols=pred):")
cm = confusion_matrix(all_trues, all_preds, labels=label_ids)
header = "          " + "  ".join(f"{l:>7}" for l in label_names)
print(header)
for i, row in enumerate(cm):
    print(f"  {label_names[i]:>5}  " + "  ".join(f"{v:7d}" for v in row))
print("=" * 60)

Running final evaluation...

FINAL EVALUATION REPORT
Accuracy  : 0.7750
Macro-F1  : 0.5319

              precision    recall  f1-score   support

         yes     0.7810    0.9224    0.8458       116
          no     0.7619    0.7385    0.7500        65
       maybe     0.0000    0.0000    0.0000        19

    accuracy                         0.7750       200
   macro avg     0.5143    0.5536    0.5319       200
weighted avg     0.7006    0.7750    0.7343       200

Confusion Matrix (rows=true, cols=pred):
              yes       no    maybe
    yes      107        9        0
     no       17       48        0
  maybe       13        6        0


---
## 14. Training history summary

In [ ]:
print(f"{'Round':>6}  {'Val Acc':>8}  {'Macro-F1':>9}  {'Threshold':>10}  {'Pseudo':>8}")
print("-" * 52)
for h in history:
    print(f"{h['round']:>6}  {h['val_acc']:>8.4f}  {h['val_f1']:>9.4f}  "
          f"{h['threshold']:>10.2f}  {h['pseudo_accepted']:>8,}")

 Round   Val Acc   Macro-F1   Threshold    Pseudo
----------------------------------------------------
     0    0.7650     0.5815        0.90         0
     1    0.7650     0.5585        0.85    27,868
     2    0.7850     0.5422        0.85    54,320
     3    0.7750     0.5319        0.80    59,752


---
## 15. Save final model

In [ ]:
save_path = cfg.OUTPUT_DIR / "final_model"
model.save(save_path)
print(f"Model saved to: {save_path}")
print("\nTo reload:")
print(f"  model = PubMedBERTClassifier.load('{save_path}')")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: outputs/final_model

To reload:
  model = PubMedBERTClassifier.load('outputs/final_model')


Answer Quality Analysis

In [ ]:
import re
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def extract_question_features(question, abstract):

    q_lower  = question.lower()
    q_tokens = q_lower.split()
    a_tokens = abstract.lower().split()

    hedging_words = [
        'may', 'might', 'could', 'possibly', 'perhaps', 'unclear',
        'uncertain', 'unknown', 'whether', 'suggest', 'appears', 'seem',
        'likely', 'unlikely', 'potential', 'possible', 'probable'
    ]

    negation_words = [
        'not', 'no', 'never', 'neither', 'nor', 'without', 'fail',
        'lack', 'absence', 'prevent', 'unable', 'cannot'
    ]

    biomedical_terms = list(set([
        'patient', 'treatment', 'disease', 'clinical', 'study',
        'efficacy', 'outcome', 'therapy', 'diagnosis', 'risk',
        'treat', 'therap', 'trial', 'drug', 'efficac', 'interven',
        'random', 'placebo', 'diagno', 'symptom', 'surviv', 'mortal',
        'adverse', 'dose', 'receptor', 'protein', 'gene', 'cell',
        'signif', 'reduc', 'increas', 'compar', 'associat', 'effect',
        'prevent', 'cancer', 'inflamm', 'immune', 'antibod', 'inhibit', 'express'
    ]))

    comparison_words = [
        'compare', 'versus', 'vs', 'differ', 'between', 'relative',
        'than', 'against', 'contrast', 'superior', 'inferior'
    ]

    features = {}


    features['q_word_count']      = len(q_tokens)
    features['a_word_count']      = len(a_tokens)
    features['q_char_count']      = len(question)
    features['q_a_length_ratio']  = len(q_tokens) / max(len(a_tokens), 1)
    features['q_avg_word_len']    = sum(len(w) for w in q_tokens) / max(len(q_tokens), 1)
    features['q_unique_word_ratio'] = len(set(q_tokens)) / max(len(q_tokens), 1)


    def _qtype(q):
        if q.startswith(('is ', 'are ', 'was ', 'were ')):
            return 'is_are'
        elif q.startswith(('does ', 'do ', 'did ')):
            return 'does_do'
        elif q.startswith(('can ', 'could ', 'should ', 'would ')):
            return 'modal'
        elif q.startswith(('what ', 'which ')):
            return 'what_which'
        elif q.startswith(('how ', 'why ')):
            return 'how_why'
        elif q.startswith('whether'):
            return 'whether'
        elif q.startswith(('has ', 'have ', 'had ')):
            return 'has_have'
        elif q.startswith(('in ', 'among ', 'for ')):
            return 'prepositional'
        else:
            return 'other'

    all_qtypes = ['is_are', 'does_do', 'modal', 'what_which',
                  'how_why', 'whether', 'has_have', 'prepositional', 'other']
    qtype = _qtype(q_lower)
    for t in all_qtypes:
        features[f'qtype_{t}'] = int(qtype == t)

    q_set   = set(q_tokens)
    a_set   = set(a_tokens)
    overlap = q_set & a_set
    union   = q_set | a_set

    features['n_shared_tokens']       = len(overlap)
    features['token_overlap_jaccard'] = len(overlap) / len(union) if union else 0
    features['q_token_coverage']      = len(overlap) / max(len(q_set), 1)


    features['n_hedging_words']    = sum(q_lower.count(w) for w in hedging_words)
    features['hedging_density']    = features['n_hedging_words'] / max(len(q_tokens), 1)
    features['n_negation_words']   = sum(q_lower.count(w) for w in negation_words)
    features['n_biomedical_terms'] = sum(q_lower.count(t) for t in biomedical_terms)
    features['biomedical_density'] = features['n_biomedical_terms'] / max(len(q_tokens), 1)
    features['n_comparison_words'] = sum(q_lower.count(w) for w in comparison_words)


    features['has_question_mark'] = int('?' in question)
    features['n_commas']          = question.count(',')
    features['n_parentheses']     = question.count('(')

    return features

In [ ]:
val_indices = val_ds.indices

all_features = []
all_questions = []
all_abstracts = []

for idx in val_indices:
    item = pqa_l[int(idx)]

    question = item['question']
    abstract = ' '.join(item['context']['contexts'])

    features = extract_question_features(question, abstract)
    all_features.append(features)
    all_questions.append(question)
    all_abstracts.append(abstract)

import pandas as pd
df_features = pd.DataFrame(all_features)

print(f"Extracted {len(df_features.columns)} features from {len(df_features)} questions")
print(f"Feature columns: {list(df_features.columns)}")

Extracted 27 features from 200 questions
Feature columns: ['q_word_count', 'a_word_count', 'q_char_count', 'q_a_length_ratio', 'q_avg_word_len', 'q_unique_word_ratio', 'qtype_is_are', 'qtype_does_do', 'qtype_modal', 'qtype_what_which', 'qtype_how_why', 'qtype_whether', 'qtype_has_have', 'qtype_prepositional', 'qtype_other', 'n_shared_tokens', 'token_overlap_jaccard', 'q_token_coverage', 'n_hedging_words', 'hedging_density', 'n_negation_words', 'n_biomedical_terms', 'biomedical_density', 'n_comparison_words', 'has_question_mark', 'n_commas', 'n_parentheses']


In [ ]:

answer_correct = [int(pred == true) for pred, true in zip(all_preds, all_trues)]
df_features['answer_correct'] = answer_correct

correct_count = sum(answer_correct)
incorrect_count = len(answer_correct) - correct_count

print(f"Labeled {len(answer_correct)} questions")
print(f"  Correct:   {correct_count} ({correct_count/len(answer_correct)*100:.1f}%)")
print(f"  Incorrect: {incorrect_count} ({incorrect_count/len(answer_correct)*100:.1f}%)")

Labeled 200 questions
  Correct:   155 (77.5%)
  Incorrect: 45 (22.5%)


In [ ]:

feature_cols = [col for col in df_features.columns if col != 'answer_correct']

correlations = []
for feature in feature_cols:
    corr = df_features[feature].corr(df_features['answer_correct'])
    correlations.append({
        'feature': feature,
        'correlation': corr,
        'abs_correlation': abs(corr)
    })

df_corr = pd.DataFrame(correlations).sort_values('abs_correlation', ascending=False)

print("\nTop 10 features correlated with correct answers:\n")
print(df_corr.head(10)[['feature', 'correlation']].to_string(index=False))
print("\n(Positive = predicts correct, Negative = predicts failure)")


Top 10 features correlated with correct answers:

              feature  correlation
   n_biomedical_terms     0.125310
        n_parentheses    -0.115815
     q_a_length_ratio     0.113919
   n_comparison_words     0.112247
   biomedical_density     0.096217
         a_word_count    -0.085995
         q_char_count     0.085252
         q_word_count     0.082563
token_overlap_jaccard     0.079774
  q_unique_word_ratio    -0.079686

(Positive = predicts correct, Negative = predicts failure)


In [ ]:

X_quality = df_features[feature_cols].values
y_quality = df_features['answer_correct'].values

print(f"\nTraining data: {X_quality.shape[0]} samples, {X_quality.shape[1]} features")
print(f"Class balance: {y_quality.mean():.1%} correct, {(1-y_quality.mean()):.1%} incorrect")


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=cfg.SEED)


predictors = {
    'Logistic Regression': LogisticRegression(random_state=cfg.SEED, max_iter=1000),
    'MLP (small)': MLPClassifier(hidden_layer_sizes=(50, 25), random_state=cfg.SEED,
                                  max_iter=500, early_stopping=True),
    'Random Forest (feature importance)': RandomForestClassifier(n_estimators=100,
                                                                   random_state=cfg.SEED)
}


results = {}

for name, predictor in predictors.items():
    print(f"\n{name}")


    start_time = __import__('time').time()

    cv_acc = cross_val_score(predictor, X_quality, y_quality, cv=cv, scoring='accuracy')
    cv_f1 = cross_val_score(predictor, X_quality, y_quality, cv=cv, scoring='f1')
    cv_auc = cross_val_score(predictor, X_quality, y_quality, cv=cv, scoring='roc_auc')

    training_time = __import__('time').time() - start_time

    predictor.fit(X_quality, y_quality)

    results[name] = {
        'cv_acc': cv_acc,
        'cv_f1': cv_f1,
        'cv_auc': cv_auc,
        'model': predictor,
        'time': training_time
    }

    print(f"    Accuracy : {cv_acc.mean():.4f} ± {cv_acc.std():.4f}")
    print(f"    F1       : {cv_f1.mean():.4f} ± {cv_f1.std():.4f}")
    print(f"    ROC-AUC  : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")
    print(f"    Time     : {training_time:.2f}s")

baseline_acc = max(y_quality.mean(), 1 - y_quality.mean())
print(f"\nMajority-class baseline: {baseline_acc:.4f}")


Training data: 200 samples, 27 features
Class balance: 77.5% correct, 22.5% incorrect

Logistic Regression
    Accuracy : 0.7550 ± 0.0292
    F1       : 0.8601 ± 0.0193
    ROC-AUC  : 0.5147 ± 0.0982
    Time     : 2.08s

MLP (small)
    Accuracy : 0.7650 ± 0.0200
    F1       : 0.8667 ± 0.0131
    ROC-AUC  : 0.4323 ± 0.1115
    Time     : 0.35s

Random Forest (feature importance)
    Accuracy : 0.7500 ± 0.0158
    F1       : 0.8570 ± 0.0103
    ROC-AUC  : 0.4065 ± 0.0816
    Time     : 2.37s

Majority-class baseline: 0.7750


In [ ]:

print(f"{'Model':<40} {'Accuracy':>12} {'F1':>10} {'ROC-AUC':>10}")

print(f"{'Majority-class baseline':<40} {baseline_acc:>12.4f} {'—':>10} {'—':>10}")

for name, res in results.items():
    print(f"{name:<40} {res['cv_acc'].mean():>12.4f} {res['cv_f1'].mean():>10.4f} {res['cv_auc'].mean():>10.4f}")

Model                                        Accuracy         F1    ROC-AUC
Majority-class baseline                        0.7750          —          —
Logistic Regression                            0.7550     0.8601     0.5147
MLP (small)                                    0.7650     0.8667     0.4323
Random Forest (feature importance)             0.7500     0.8570     0.4065


In [ ]:
print("FEATURE IMPORTANCE ANALYSIS (Logistic Regression coefficients)")

lr_model = results['Logistic Regression']['model']
coefficients = lr_model.coef_[0]

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': coefficients
}).sort_values('coefficient', key=abs, ascending=False)

print("\n  Top 10 most influential features:")
print(f"  {'Feature':<30} {'Coefficient':>12} {'Direction':>12}")
print("  " + "-" * 57)

for _, row in feature_importance.head(10).iterrows():
    direction = '+correct' if row['coefficient'] > 0 else '−correct'
    print(f"  {row['feature']:<30} {row['coefficient']:>12.4f} {direction:>12}")

print("\n  Random Forest feature importance (top 10):")
print(f"  {'Feature':<30} {'Importance':>12}")
print("  " + "-" * 42)

rf_model = results['Random Forest (feature importance)']['model']
rf_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

for _, row in rf_importance.head(10).iterrows():
    print(f"  {row['feature']:<30} {row['importance']:>12.4f}")

FEATURE IMPORTANCE ANALYSIS (Logistic Regression coefficients)

  Top 10 most influential features:
  Feature                         Coefficient    Direction
  ---------------------------------------------------------
  n_parentheses                       -0.8849     −correct
  n_comparison_words                   0.8169     +correct
  n_hedging_words                      0.5835     +correct
  n_commas                             0.5414     +correct
  n_biomedical_terms                   0.2893     +correct
  q_unique_word_ratio                 -0.1714     −correct
  q_avg_word_len                       0.1693     +correct
  qtype_has_have                       0.1567     +correct
  qtype_is_are                         0.1521     +correct
  q_word_count                         0.1500     +correct

  Random Forest feature importance (top 10):
  Feature                          Importance
  ------------------------------------------
  q_avg_word_len                       0.1134
  a_word

In [ ]:
print("PER-QUESTION-TYPE CORRECTNESS ANALYSIS")

def get_question_type(question):
    q_lower = question.lower().strip()
    if q_lower.startswith(('is ', 'are ', 'was ', 'were ')):
        return 'is_are'
    elif q_lower.startswith(('does ', 'do ', 'did ')):
        return 'does_do'
    elif q_lower.startswith(('can ', 'could ', 'should ', 'would ')):
        return 'modal'
    elif q_lower.startswith(('what ', 'which ')):
        return 'what_which'
    elif q_lower.startswith(('how ', 'why ')):
        return 'how_why'
    elif q_lower.startswith('whether'):
        return 'whether'
    elif q_lower.startswith(('has ', 'have ', 'had ')):
        return 'has_have'
    elif q_lower.startswith(('in ', 'among ', 'for ')):
        return 'prepositional'
    else:
        return 'other'

df_features['question_type'] = [get_question_type(q) for q in all_questions]

type_stats = df_features.groupby('question_type')['answer_correct'].agg(['count', 'mean'])
type_stats.columns = ['N', 'Correct rate']
type_stats = type_stats.sort_values('Correct rate', ascending=False)

print(f"\n  {'Question type':<20} {'N':>5} {'Correct rate':>15}")
print("  " + "-" * 55)
for qtype, row in type_stats.iterrows():
    print(f"  {qtype:<20} {int(row['N']):>5} {row['Correct rate']:>15.3f}")

PER-QUESTION-TYPE CORRECTNESS ANALYSIS

  Question type            N    Correct rate
  -------------------------------------------------------
  has_have                 1           1.000
  is_are                  54           0.796
  other                   68           0.779
  modal                   17           0.765
  does_do                 60           0.750


In [ ]:
print("FEATURE DISTRIBUTIONS: CORRECT vs INCORRECT")

key_features = [
    'token_overlap_jaccard',
    'q_word_count',
    'n_hedging_words',
    'biomedical_density',
    'n_comparison_words'
]

for feature in key_features:
    correct_vals = df_features[df_features['answer_correct'] == 1][feature]
    incorrect_vals = df_features[df_features['answer_correct'] == 0][feature]

    print(f"\n  {feature}")
    print(f"    Correctly predicted   — mean: {correct_vals.mean():.4f}, std: {correct_vals.std():.4f}")
    print(f"    Incorrectly predicted — mean: {incorrect_vals.mean():.4f}, std: {incorrect_vals.std():.4f}")

FEATURE DISTRIBUTIONS: CORRECT vs INCORRECT

  token_overlap_jaccard
    Correctly predicted   — mean: 0.0637, std: 0.0290
    Incorrectly predicted — mean: 0.0583, std: 0.0268

  q_word_count
    Correctly predicted   — mean: 12.5742, std: 4.4866
    Incorrectly predicted — mean: 11.7333, std: 3.3466

  n_hedging_words
    Correctly predicted   — mean: 0.0452, std: 0.2083
    Incorrectly predicted — mean: 0.0222, std: 0.1491

  biomedical_density
    Correctly predicted   — mean: 0.0884, std: 0.0921
    Incorrectly predicted — mean: 0.0675, std: 0.0867

  n_comparison_words
    Correctly predicted   — mean: 0.1677, std: 0.5073
    Incorrectly predicted — mean: 0.0444, std: 0.2084
